# Cat vs Dog Classification with Transfer Learning
This notebook walks through the dataset, training pipeline, and evaluation results for the Cats vs Dogs image classification project.

In [6]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
sys.path.append(str(project_root / 'src'))

import matplotlib.pyplot as plt
from data_loader import get_class_counts
from torchvision.utils import make_grid
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

## Environment and Dataset Overview
The dataset is stored in `../Dataset` and contains the two classes `Cat` and `Dog`. We use transfer learning with a ResNet18 backbone trained on ImageNet.

In [ ]:
data_dir = project_root / 'Dataset'
counts = get_class_counts(data_dir)
print('Dataset root:', data_dir)
print('Class counts:')
for label, count in counts.items():
    print(f'  {label}: {count}')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
dataset = ImageFolder(str(data_dir), transform=transform)
loader = DataLoader(dataset, batch_size=16, shuffle=True)
images, labels = next(iter(loader))
grid = make_grid(images[:8], nrow=4, normalize=True, pad_value=0.2)
plt.figure(figsize=(10, 5))
plt.imshow(grid.permute(1, 2, 0).cpu())
plt.title('Sample images from the dataset')
plt.axis('off')
plt.show()
print('Classes:', dataset.classes)

## Training the model
The next step is to train the model using the implementation in `src/train.py`. The script performs dataset splitting, transfer learning, and logs training performance.

In [ ]:
!{sys.executable} ../src/train.py --data-dir ../Dataset --output-dir ../results --epochs 3 --batch-size 32 --feature-extract

## Evaluation
After training, evaluate the best model on the held-out test split and inspect the confusion matrix.

In [ ]:
!python ../src/evaluate.py --data-dir ../Dataset --weights ../results/best_model.pth --output-dir ../results

## Inspect model
Observe the layer names and weight shapes.


In [ ]:
import torch

checkpoint = torch.load('../results/best_model.pth', map_location=torch.device('cpu'))

print(type(checkpoint))
print(checkpoint.keys())

## Notes
- The notebook demonstrates dataset loading, sample visualization, training invocation, and evaluation.
- See the saved evaluation artifacts in `results/`.